# Palantir Cell Trajectory
**Kernel:** Python (palantir-venv)  
**Data:** `./data/Processed/CD8_DE2500_singleVDJ_emb.h5mu` — CD8 T cells, 14 957 cells × 2 500 HVGs, log-normalised.

**Subtypes:** CD8_Mitosis → CD8_Teff / CD8_Trm_exh_{L,M,H}  
**Start cell:** CD8_Mitosis edge cell (extreme DC1, proliferating progenitor).  
**Terminal states:** CD8_Teff (cytotoxic terminal), CD8_Trm_exh_H (deep exhaustion).

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import h5py
import anndata as ad
import scanpy as sc
import palantir

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, frameon=False)
print('palantir', palantir.__version__)
print('scanpy  ', sc.__version__)
print('anndata ', ad.__version__)

## 1. Load GEX from h5mu
The venv does not include `muon`; we read the `gex` modality directly with `h5py`.

In [ ]:
path = r"/ix1/ylee/Yifan_Zhang/Code_data/Tumor/GSE139555_2019/data/Processed/"
filename = "CD8_DE2500_singleVDJ_emb.h5mu"
# filename = "T_DE_per_sample_TCRemb.h5mu"

DATA_PATH = path + filename

In [ ]:
import h5py

for path in [path+"CD8_DE2500_singleVDJ_emb.h5mu", path+"T_DE_per_sample_TCRemb.h5mu"]:
    with h5py.File(path, "r") as f:
        print(f"\n=== {path} ===")
        # top-level anndata/mudata version
        print("root attrs:", dict(f.attrs))
        
        xg = f["mod"]["gex"]["X"]
        print("X type:", type(xg))
        print("X attrs:", dict(xg.attrs))
        
        # if X is a group (CSR), check subkeys
        if hasattr(xg, 'keys'):
            print("X keys:", list(xg.keys()))

In [ ]:
# import muon as mu

# mdata_ori = mu.read(DATA_PATH)
# mdata = mdata_ori.copy()
# mdata

In [ ]:
# works for "CD8_DE2500_singleVDJ_emb.h5mu"

# def load_gex_from_h5mu(path: str) -> ad.AnnData:
#     """Read the 'gex' modality from an h5mu file without muon."""
#     with h5py.File(path, "r") as f:
#         g = f["mod"]["gex"]

#         # ── X (CSR) ──────────────────────────────────────────────
#         xg = g["X"]
#         attrs = dict(xg.attrs)

#         shape = tuple(xg.shape)
        
#         X = sp.csr_matrix(
#             (xg["data"][:], xg["indices"][:], xg["indptr"][:]),
#             shape=shape,
#         )

#         # ── obs ──────────────────────────────────────────────────
#         obs_idx = [s.decode() for s in g["obs"]["_index"][:]]
#         obs_dict = {}
#         for col in g["obs"].keys():
#             if col == "_index":
#                 continue
#             col_data = g["obs"][col]
#             if "categories" in col_data:
#                 cats = [s.decode() for s in col_data["categories"][:]]
#                 codes = col_data["codes"][:]
#                 obs_dict[col] = pd.Categorical.from_codes(codes, cats)
#             else:
#                 vals = col_data[:]
#                 if vals.dtype.kind in ("S", "O"):
#                     vals = [v.decode() if isinstance(v, bytes) else str(v) for v in vals]
#                 obs_dict[col] = vals
#         obs = pd.DataFrame(obs_dict, index=obs_idx)

#         # ── var ──────────────────────────────────────────────────
#         var_idx = [s.decode() for s in g["var"]["_index"][:]]
#         var_dict = {}
#         for col in g["var"].keys():
#             if col == "_index":
#                 continue
#             col_data = g["var"][col]
#             try:
#                 if "categories" in col_data:
#                     cats = [s.decode() for s in col_data["categories"][:]]
#                     codes = col_data["codes"][:]
#                     var_dict[col] = pd.Categorical.from_codes(codes, cats)
#                 else:
#                     v = col_data[:]
#                     if v.dtype.kind in ("S", "O"):
#                         v = [x.decode() if isinstance(x, bytes) else str(x) for x in v]
#                     var_dict[col] = v
#             except Exception:
#                 pass
#         var = pd.DataFrame(var_dict, index=var_idx)

#         # ── obsm ─────────────────────────────────────────────────
#         obsm = {k: g["obsm"][k][:] for k in g["obsm"].keys()}

#     adata = ad.AnnData(X=X, obs=obs, var=var)
#     for k, v in obsm.items():
#         adata.obsm[k] = v
#     return adata

In [ ]:
def load_gex_from_h5mu(path: str) -> ad.AnnData:
    """Read the 'gex' modality from an h5mu file without muon."""

    def read_dataframe(grp):
        idx_col = grp.attrs["_index"]
        raw_idx = grp[idx_col][:]
        index = [s.decode() if isinstance(s, bytes) else s for s in raw_idx]
        d = {}
        for col in grp.keys():
            if col == idx_col:
                continue
            col_data = grp[col]
            try:
                if isinstance(col_data, h5py.Group) and "categories" in col_data:
                    cats = [s.decode() if isinstance(s, bytes) else s
                            for s in col_data["categories"][:]]
                    codes = col_data["codes"][:]
                    d[col] = pd.Categorical.from_codes(codes, cats)
                elif isinstance(col_data, h5py.Dataset):
                    vals = col_data[:]
                    if vals.dtype.kind in ("S", "O"):
                        vals = [v.decode() if isinstance(v, bytes) else str(v)
                                for v in vals]
                    d[col] = vals
            except Exception:
                pass
        return pd.DataFrame(d, index=index)

    with h5py.File(path, "r") as f:
        g = f["mod"]["gex"]

        # ── X ────────────────────────────────────────────────────
        xg    = g["X"]
        attrs = dict(xg.attrs)
        enc   = attrs.get("encoding-type", "")

        if enc in ("csr_matrix", "csc_matrix"):
            if "shape" in attrs:
                shape = tuple(int(x) for x in attrs["shape"])
            else:
                n_obs  = xg["indptr"].shape[0] - 1
                n_vars = g["var"][g["var"].attrs["_index"]].shape[0]
                shape  = (n_obs, n_vars)
            X = sp.csr_matrix(
                (xg["data"][:], xg["indices"][:], xg["indptr"][:]),
                shape=shape,
            )
        else:                           # dense array
            X = sp.csr_matrix(xg[:])

        # ── obs / var ─────────────────────────────────────────────
        obs  = read_dataframe(g["obs"])
        var  = read_dataframe(g["var"])

        # ── obsm ──────────────────────────────────────────────────
        obsm = {}
        for k in g.get("obsm", {}).keys():
            item = g["obsm"][k]
            if isinstance(item, h5py.Dataset):
                obsm[k] = item[:]

    adata = ad.AnnData(X=X, obs=obs, var=var)
    for k, v in obsm.items():
        adata.obsm[k] = v
    return adata

In [ ]:
adata_ori = load_gex_from_h5mu(DATA_PATH)
adata = adata_ori.copy()

In [ ]:
# sc.pp.highly_variable_genes(adata, n_top_genes=500, batch_key="sample")
# adata = adata[:, adata.var['highly_variable']].copy()

In [ ]:
# adata = adata[~adata.obs['subtype'].isin(['CD8_Mitosis'])]

In [ ]:
adata

## 2. Quick sanity check — existing UMAP

In [ ]:
import scanpy.external as sce
import harmonypy as hm

if "X_pca_harmony" not in adata.obsm:
    del adata.obsm["X_pca"]
    sc.pp.pca(adata, n_comps=50)
    
    pca_matrix = np.array(adata.obsm["X_pca"], dtype=np.float64)
    meta_data = adata.obs[['patient']]

    ho = hm.run_harmony(
        pca_matrix, 
        meta_data, 
        ['patient'],
        max_iter_harmony=20
    )
    adata.obsm["X_pca_harmony"] = ho.Z_corr
    sc.pp.neighbors(adata, use_rep="X_pca_harmony", key_added="neighbors_harmony")

    # Compute and save UMAP from harmony neighbors
    sc.tl.umap(adata, neighbors_key="neighbors_harmony")
    adata.obsm["X_umap_harmony"] = adata.obsm["X_umap"].copy()

    # sce.pp.harmony_integrate(adata, key=['patient'],  basis='X_pca', max_iter_harmony=20, 
    #                           # theta = 4, lamb = 1.5, sigma=0.1, tau=1
    #                         )

In [ ]:
adata

In [ ]:
sc.pp.neighbors(adata, n_neighbors=10, use_rep='X_pca_harmony')
sc.tl.leiden(adata)

In [ ]:
sc.pl.embedding(
    adata, basis="X_umap_harmony",
    color=["subtype", "source", "patient", "leiden"],
    ncols=2, wspace=0.4, frameon=False,
)

## 3. Palantir preprocessing — diffusion maps
Data is already log-normalised (`uns['log1p']`) and PCA is stored in `obsm['X_pca']`.  
Palantir expects a PCA-reduced representation as input.

In [ ]:
# Use stored PCA.  Palantir will compute its own diffusion maps from it.
pca_proj = adata.obsm["X_pca_harmony"]
print(f"PCA shape: {pca_proj.shape}")

# Build a temporary AnnData with PCA as X for Palantir's diffusion-map step
adata_pca = ad.AnnData(
    X=pca_proj.astype(np.float64),
    obs=adata.obs.copy(),
)
adata_pca.obsm["X_pca"] = pca_proj.astype(np.float64)
adata_pca.obsm["X_umap"] = adata.obsm["X_umap"]

# Compute diffusion maps
palantir.utils.run_diffusion_maps(
    adata_pca,
    pca_key="X_pca",
    n_components=20,
    knn=15,
    alpha=0.8
)
print("Diffusion map keys:", list(adata_pca.obsm.keys()))

In [ ]:
print(adata_pca.uns['DM_EigenValues'])

In [ ]:
adata_pca

In [ ]:
# sc.pp.neighbors(adata_pca)
# sc.tl.umap(adata_pca)


In [ ]:
# %matplotlib widget
# import matplotlib
# matplotlib.use('TkAgg')
# import matplotlib.pyplot as plt
# import scanpy as sc
# import numpy as np

# fig, ax = plt.subplots()
# sc.pl.umap(adata, color='cell_type', ax=ax, show=False)

# def onclick(event):
#     x, y = event.xdata, event.ydata
#     if x is None:
#         return
#     coords = adata.obsm['X_umap']
#     dist = np.sqrt((coords[:,0]-x)**2 + (coords[:,1]-y)**2)
#     idx = np.argmin(dist)
#     print(f"Cell: {adata.obs_names[idx]}")
#     plt.close(fig)  # closes window after click

# fig.canvas.mpl_connect('button_press_event', onclick)
# plt.show()  # blocks until window is closed

In [ ]:
# Multiscale space (used as the actual distance space for Palantir)
palantir.utils.determine_multiscale_space(adata_pca)
print("Multiscale space shape:", adata_pca.obsm["DM_EigenVectors_multiscaled"].shape)


In [ ]:
# Impute genes

# imputed_X = palantir.utils.run_magic_imputation(adata_pca)
# adata_pca

In [ ]:
# Palantir stores diffusion eigenvectors in "DM_EigenVectors" (cols 0..n-1, no trivial component)
dm = adata_pca.obsm["DM_EigenVectors"]
for i in range(dm.shape[1]):
    adata_pca.obs[f"DC{i+1}"] = dm[:, i]

# min(4, dm.shape[1])

sc.pl.embedding(
    adata_pca, basis="X_umap",
    color=["subtype", 'leiden', "DC1", "DC2", "DC3"],
    ncols=2, wspace=0.4, frameon=False,
    color_map="RdBu_r",
)

## 4. Start & terminal cell selection  ---- 1 terminal
Pick a **CD8_Mitosis** cell at the edge of the cluster (extreme DC1) as the root,
and a **CD8_Trm_exh_H** cell at the deep-exhaustion edge as the terminal state.


In [ ]:
# Start cell: CD8_Mitosis cell most extreme along DC1 (edge of cluster)
# mit_mask = adata_pca.obs['subtype'] == 'CD8_Mitosis'
mit_mask = adata_pca.obs['subtype'] == 'CD8_Teff'
print(f'CD8_Mitosis cells: {mit_mask.sum()}')

if mit_mask.sum() == 0:
    raise ValueError('No CD8_Mitosis cells found — check subtype labels')

dc1_mit = adata_pca.obs.loc[mit_mask, 'DC1']
# Edge = minimum DC1 in this cluster (adjust to idxmax if orientation is reversed)
start_cell = dc1_mit.idxmin()

# Terminal cell: CD8_Trm_exh_H cell most extreme along DC1
exh_mask = adata_pca.obs['subtype'] == 'CD8_Trm_exh_H'

if exh_mask.sum() == 0:
    raise ValueError('No CD8_Trm_exh_H cells found — check subtype labels')

dc1_exh = adata_pca.obs.loc[exh_mask, 'DC1']
terminal_cell = dc1_exh.idxmax()


In [ ]:
# ── Start cell: CD8_Mitosis ──────────────────────────────────────────────────
mit_mask_leiden = mit_mask & (adata_pca.obs['leiden'] == '3')  # replace '3'
start_cell = adata_pca.obs.loc[mit_mask_leiden, 'DC1'].idxmax()

# ── Terminal cell: CD8_Trm_exh_H ────────────────────────────────────────────
exh_mask_leiden = exh_mask & (adata_pca.obs['leiden'] == '1')  # replace '7'
terminal_cell = adata_pca.obs.loc[exh_mask_leiden, 'DC1'].idxmin()


In [ ]:
adata_pca.obs['is_start']    = adata_pca.obs_names == start_cell
adata_pca.obs['is_terminal'] = adata_pca.obs_names == terminal_cell

fig, ax = plt.subplots(figsize=(7, 5))
sc.pl.embedding(adata_pca, basis='X_umap', color='subtype',
                ax=ax, show=False, frameon=False)

xy = adata_pca.obsm['X_umap'][adata_pca.obs_names == start_cell][0]
ax.scatter(*xy, s=120, marker='*', color='black', zorder=5, label='Start (CD8_Mitosis)')

xy = adata_pca.obsm['X_umap'][adata_pca.obs_names == terminal_cell][0]
ax.scatter(*xy, s=120, marker='*', color='red', zorder=5, label='Terminal (CD8_Trm_exh_H)')

ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', frameon=False)
plt.tight_layout()
plt.show()

## 5. Run Palantir

In [ ]:
pr_res = palantir.core.run_palantir(
    adata_pca,
    early_cell=start_cell,
    terminal_states=[terminal_cell],
    use_early_cell_as_start=True,
    num_waypoints=1200,
)
print(pr_res)


## 6. Visualise Palantir results

In [ ]:
# Copy pseudotime and entropy back for scanpy plotting
adata_pca.obs["palantir_pseudotime"] = pr_res.pseudotime
adata_pca.obs["palantir_entropy"]    = pr_res.entropy

sc.pl.embedding(
    adata_pca, basis="X_umap",
    color=["subtype", "palantir_pseudotime", "palantir_entropy"],
    ncols=3, wspace=0.4, frameon=False,
    color_map="magma",
)

In [ ]:
# # Per-terminal-state fate probabilities
# for term in pr_res.branch_probs.columns:
#     col = f"fate_{term}"
#     adata_pca.obs[col] = pr_res.branch_probs[term].values

# fate_cols = [c for c in adata_pca.obs.columns if c.startswith("fate_")]
# sc.pl.embedding(
#     adata_pca, basis="X_umap",
#     color=fate_cols,
#     ncols=min(3, len(fate_cols)),
#     wspace=0.4, frameon=False,
#     color_map="Blues",
#     vmin=0, vmax=1,
# )

In [ ]:
# Palantir built-in result plot
# palantir.plot.plot_palantir_results(adata_pca, s=3)

## 7. Pseudotime distribution by subtype

In [ ]:
import seaborn as sns

order = (
    adata_pca.obs.groupby("subtype")["palantir_pseudotime"]
    .median()
    .sort_values()
    .index.tolist()
)

fig, ax = plt.subplots(figsize=(7, 3.5))
sns.violinplot(
    data=adata_pca.obs,
    x="subtype", y="palantir_pseudotime",
    order=order, cut=0, inner="box", ax=ax,
)
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha="right", fontsize=9)
ax.set_title("Palantir pseudotime per CD8 subtype")
ax.set_xlabel("")
plt.tight_layout()
plt.show()

###  Two terminals

In [ ]:
# ── Start cell: CD8_Mitosis ──────────────────────────────────────────────────
# mit_mask = adata_pca.obs['subtype'] == 'CD8_Mitosis'

mit_mask_leiden = mit_mask & (adata_pca.obs['leiden'] == '10')  # replace '3'
start_cell = adata_pca.obs.loc[mit_mask_leiden, 'DC1'].idxmax()

# ── Terminal cells ───────────────────────────────────────────────────────────
terminal_configs = {
    'CD8_Trm_exh_L': {'leiden': '0',  'extreme': 'idxmin'},  # replace leiden
    'CD8_Teff':      {'leiden': '1',  'extreme': 'idxmin'},  # replace leiden
}

terminal_cells = {}
for subtype, cfg in terminal_configs.items():
    mask = adata_pca.obs['subtype'] == subtype

    mask_leiden = mask & (adata_pca.obs['leiden'] == cfg['leiden'])
    dc1 = adata_pca.obs.loc[mask_leiden, 'DC1']
    cell = getattr(dc1, cfg['extreme'])()
    terminal_cells[subtype] = cell

terminal_states = list(terminal_cells.values())

In [ ]:
# ── Mark start and terminal cells in obs ────────────────────────────────────
adata_pca.obs['is_start']    = adata_pca.obs_names == start_cell
adata_pca.obs['is_terminal'] = adata_pca.obs_names.isin(terminal_states)

# ── Plot all in one figure ───────────────────────────────────────────────────
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
sc.pl.embedding(adata_pca, basis='X_umap', color='subtype',
                ax=ax, show=False, frameon=False)

# Start cell
xy = adata_pca.obsm['X_umap'][adata_pca.obs_names == start_cell][0]
ax.scatter(*xy, s=120, marker='*', color='black', zorder=5, label='Start (CD8_Mitosis)')

# Terminal cells
colors = ['red', 'blue', 'green', 'orange', 'purple']
for (subtype, cell), color in zip(terminal_cells.items(), colors):
    xy = adata_pca.obsm['X_umap'][adata_pca.obs_names == cell][0]
    ax.scatter(*xy, s=120, marker='*', color=color, zorder=5, label=f'Terminal ({subtype})')

ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', frameon=False)
plt.tight_layout()
plt.show()


In [ ]:
pr_res_2 = palantir.core.run_palantir(
    adata_pca,
    early_cell=start_cell,
    terminal_states=terminal_states,
    use_early_cell_as_start=True,
    num_waypoints=1200,
)

In [ ]:
# ── Save results to adata_pca.obs ────────────────────────────────────────────
adata_pca.obs['palantir_pseudotime_2'] = pr_res_2.pseudotime
adata_pca.obs['palantir_entropy_2']    = pr_res_2.entropy

for col in pr_res_2.branch_probs.columns:
    clean_name = f"fate_{col.replace(' ', '_')}"
    adata_pca.obs[clean_name] = pr_res_2.branch_probs[col]

fate_cols = [c for c in adata_pca.obs.columns if c.startswith('fate_')]
print("Saved columns:", ['palantir_pseudotime', 'palantir_entropy'] + fate_cols)

plot_cols = ['palantir_pseudotime_2', 'palantir_entropy_2']
n_plots   = len(plot_cols)
fig, axes = plt.subplots(1, n_plots, figsize=(5 * n_plots, 4))

for ax, col in zip(axes, plot_cols):
    sc.pl.embedding(adata_pca, basis='X_umap', color=col,
                    ax=ax, show=False, frameon=False,
                    color_map='RdYlBu_r')
    ax.set_title(col.replace('_', ' '))

plt.tight_layout()
plt.show()

In [ ]:
adata_pca.obsm['palantir_fate_probabilities']

### save 

In [ ]:
new_filename = filename.replace('.h5mu', '_pseudo.h5ad')
adata_pca.write_h5ad(path + new_filename)

In [ ]:
new_filename

## 8. CellRank — Transition Matrix & Macrostates
Uses **CellRank 2** (`PseudotimeKernel` + `ConnectivityKernel` → `GPCCA`) to identify
macrostates, predict terminal states, compute fate probabilities, and find lineage-driver genes.
The Palantir pseudotime computed above is used as the time ordering.

In [ ]:
import cellrank as cr
from cellrank.kernels import PseudotimeKernel, ConnectivityKernel
from cellrank.estimators import GPCCA

print('cellrank', cr.__version__)


In [ ]:
# PseudotimeKernel: transition probabilities derived from Palantir pseudotime
pk = PseudotimeKernel(adata_pca, time_key="palantir_pseudotime")
pk.compute_transition_matrix()
pk


In [ ]:
pk.plot_random_walks(
    seed=0,
    n_sims=100,
    start_ixs={'subtype': 'CD8_Mitosis'},
    basis='X_umap',
    max_iter=200,
    legend_loc='on data',
    color='subtype',
)


In [ ]:
# ConnectivityKernel: uses the kNN graph from sc.pp.neighbors
ck = ConnectivityKernel(adata_pca).compute_transition_matrix()

# Combine: 80 % pseudotime-directed + 20 % graph connectivity
combined_kernel = 0.8 * pk + 0.2 * ck
combined_kernel


In [ ]:
g = GPCCA(combined_kernel)
g.compute_schur(n_components=15)


In [ ]:
g.plot_spectrum(real_only=True)


In [ ]:
# n_states chosen to cover CD8 subtypes; adjust based on the eigenvalue spectrum above
g.compute_macrostates(n_states=6, cluster_key="subtype")
g.plot_macrostates(which="all", basis="X_umap", legend_loc="right")


In [ ]:
g.predict_terminal_states(method="top_n", n_states=3)
g.plot_macrostates(which="terminal", basis="X_umap", legend_loc="right")


In [ ]:
g.compute_fate_probabilities()
g.plot_fate_probabilities(legend_loc="right", basis="X_umap")


In [ ]:
cr.pl.circular_projection(adata_pca, keys="subtype", legend_loc="right")


In [ ]:
# Write transition matrix + GPCCA results (branch_masks, lineages_fwd, etc.) into adata_pca
pk.write_to_adata(key="T_fwd")
# g.write_to_adata()   # writes branch_masks, lineages_fwd, term-state obs cols

# Copy all CellRank obsm/uns keys from adata_pca -> adata (full GEX).
# We copy lineages_*, branch_masks, T_fwd so downstream cr.pl calls work on adata.
_CR_OBS_PREFIXES = ("lineages", "branch", "T_fwd")
pos = adata_pca.obs_names.get_indexer(adata.obs_names)  # int positions
for _k in list(adata_pca.obsm):
    if any(_k.startswith(p) for p in _CR_OBS_PREFIXES):
        adata.obsm[_k] = adata_pca.obsm[_k][pos]
for _k in list(adata_pca.uns):
    if any(_k.startswith(p) for p in _CR_OBS_PREFIXES) or _k in ("lineages_fwd_colors",):
        adata.uns[_k] = adata_pca.uns[_k]
# Also copy terminal/macrostate obs columns so cr.pl can colour cells
for _col in adata_pca.obs.columns:
    if any(_col.startswith(p) for p in ("terminal", "macrostate", "lineage")):
        adata.obs[_col] = adata_pca.obs[_col].reindex(adata.obs_names).values

terminal_lineages = list(g.terminal_states.cat.categories)
print("Terminal lineages:", terminal_lineages)

driver_tables = {}
for lineage in terminal_lineages:
    try:
        drivers = g.compute_lineage_drivers(
            lineages=lineage,
            return_drivers=True,
        )
        driver_tables[lineage] = drivers
        print(f'\n=== Top drivers -> {lineage} ===')
        print(drivers.head(10).to_string())
    except Exception as exc:
        print(f'Skipped {lineage}: {exc}')


In [ ]:
# Gene expression trends along each lineage (CellRank GAM model)
# cr.models.GAM is built on adata (full gene expression), not adata_pca.
cd8_genes = [
    "TCF7", "LEF1", "SELL",            # stem / memory
    "GZMB", "PRF1", "CX3CR1",          # effector / cytotoxic
    "PDCD1", "HAVCR2", "TOX", "ENTPD1", # exhaustion
]
cd8_genes = [g for g in cd8_genes if g in adata.var_names]
print("Genes available for trends:", cd8_genes)

if cd8_genes:
    adata.obs["palantir_pseudotime"] = (
        adata_pca.obs["palantir_pseudotime"].reindex(adata.obs_names)
    )

    model = cr.models.GAM(adata)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        cr.pl.gene_trends(
            adata,
            model=model,
            genes=cd8_genes[:6],
            same_plot=True,
            ncols=2,
            time_key="palantir_pseudotime",
            hide_cells=True,
        )


In [ ]:
if cd8_genes:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        cr.pl.heatmap(
            adata,
            model=model,
            genes=cd8_genes,
            lineages=terminal_lineages,
            time_key="palantir_pseudotime",
            cbar=False,
            show_all_genes=True,
        )


## 9. Gene expression trends along pseudotime
Palantir can model smooth gene expression trends with GAMs.

In [ ]:
# Copy Palantir results into the full GEX adata for trend computation
adata.obs["palantir_pseudotime"] = pr_res.pseudotime.reindex(adata.obs_names)
adata.obs["palantir_entropy"]    = pr_res.entropy.reindex(adata.obs_names)

# Store fate probabilities so select_branch_cells can build branch_masks
adata.obsm["palantir_fate_probabilities"] = pr_res.branch_probs.reindex(adata.obs_names)
palantir.presults.select_branch_cells(adata, eps=0.0, q=0.01)

for term in pr_res.branch_probs.columns:
    adata.obs[f"fate_{term}"] = pr_res.branch_probs[term].reindex(adata.obs_names).values

genes_of_interest = [
    "TCF7",   # stem-like memory
    "LEF1",   # memory
    "SELL",   # naive / central memory
    "CX3CR1", # terminally differentiated effector
    "GZMB",   # cytotoxic effector
    "PRF1",   # cytotoxic
    "PDCD1",  # exhaustion (PD-1)
    "HAVCR2", # exhaustion (TIM-3)
    "TOX",    # exhaustion TF
    "ENTPD1", # exhausted CD39
]
genes_present = [g for g in genes_of_interest if g in adata.var_names]
print("Genes found:", genes_present)
print("branch_masks shape:", adata.obsm["branch_masks"].shape)


In [ ]:
if genes_present:
    # Palantir 1.4.4 computes trends for every gene in adata;
    # subset to the genes of interest to keep runtime manageable.
    adata_trends = adata[:, genes_present].copy()
    gene_trends = palantir.presults.compute_gene_trends(
        adata_trends,
        pseudo_time_key="palantir_pseudotime",
    )
    print("Branches:", list(gene_trends.keys()))


In [ ]:
if genes_present:
    palantir.plot.plot_gene_trends(gene_trends, genes=genes_present[:6])
    plt.tight_layout()
    plt.show()

## 10. Save results

In [ ]:
# Save pseudotime / entropy / fate back into the GEX AnnData
adata.write_h5ad(path + "CD8_palantir.h5ad")